In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS silver.purchase_order
(
    purchase_order_id INT,

    partner_id INT,
    partner_name STRING,

    company_id INT,
    company_name STRING,

    currency_id INT,
    currency_name STRING,

    user_id INT,
    user_name STRING,

    payment_term_id INT,
    payment_term_name STRING,

    incoterm_id INT,
    incoterm_name STRING,

    fiscal_position_id INT,
    fiscal_position_name STRING,

    picking_type_id INT,
    picking_type_name STRING,

    invoice_count INT,

    order_name STRING,
    priority STRING,
    origin STRING,
    partner_ref STRING,

    order_state STRING,
    invoice_status STRING,
    receipt_status STRING,

    note STRING,

    amount_untaxed DOUBLE,
    amount_tax DOUBLE,
    amount_total DOUBLE,
    amount_total_cc DOUBLE,

    currency_rate DOUBLE,

    locked BOOLEAN,
    acknowledged BOOLEAN,
    receipt_reminder_email BOOLEAN,

    date_order TIMESTAMP,
    date_approve TIMESTAMP,
    date_planned TIMESTAMP,
    effective_date TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- =====================================================
-- MERGE
-- =====================================================

MERGE INTO silver.purchase_order AS target

USING
(
    SELECT
        cast( po.id As int) AS purchase_order_id ,

        cast( GET_JSON_OBJECT(po.partner_id, '$[0]') As int) AS partner_id,
        GET_JSON_OBJECT(po.partner_id, '$[1]') AS partner_name,

        cast( GET_JSON_OBJECT(po.company_id, '$[0]') As int) AS company_id,
        GET_JSON_OBJECT(po.company_id, '$[1]') AS company_name,

        cast( GET_JSON_OBJECT(po.currency_id, '$[0]') As int) AS currency_id,
        GET_JSON_OBJECT(po.currency_id, '$[1]') AS currency_name,

        cast( GET_JSON_OBJECT(po.user_id, '$[0]') As int) AS user_id,
        GET_JSON_OBJECT(po.user_id, '$[1]') AS user_name,

        cast( GET_JSON_OBJECT(po.payment_term_id, '$[0]') As int) AS payment_term_id,
        GET_JSON_OBJECT(po.payment_term_id, '$[1]') AS payment_term_name,

        cast( GET_JSON_OBJECT(po.incoterm_id, '$[0]') As int) AS incoterm_id,
        GET_JSON_OBJECT(po.incoterm_id, '$[1]') AS incoterm_name,

        cast( GET_JSON_OBJECT(po.fiscal_position_id, '$[0]') As int) AS fiscal_position_id,
        GET_JSON_OBJECT(po.fiscal_position_id, '$[1]') AS fiscal_position_name,

        cast( GET_JSON_OBJECT(po.picking_type_id, '$[0]') As int) AS picking_type_id,
        GET_JSON_OBJECT(po.picking_type_id, '$[1]') AS picking_type_name,

        CAST(po.invoice_count AS INT) AS invoice_count,

        po.order_name AS order_name, 
        po.priority,
        po.origin,
        po.partner_ref,

        po.order_state AS order_state,
        po.invoice_status,
        po.receipt_status,

        po.note,

        CAST(po.amount_untaxed AS DOUBLE) AS amount_untaxed,
        CAST(po.amount_tax AS DOUBLE) AS amount_tax,
        CAST(po.amount_total AS DOUBLE) AS amount_total,
        CAST(po.amount_total_cc AS DOUBLE) AS amount_total_cc,

        CAST(po.currency_rate AS DOUBLE) AS currency_rate,

        CAST(po.locked AS BOOLEAN) AS locked,
        CAST(po.acknowledged AS BOOLEAN) AS acknowledged,
        CAST(po.receipt_reminder_email AS BOOLEAN) AS receipt_reminder_email,

        CAST(po.date_order AS TIMESTAMP) AS date_order,
        CAST(po.date_approve AS TIMESTAMP) AS date_approve,
        CAST(po.date_planned AS TIMESTAMP) AS date_planned,
        CAST(po.effective_date AS TIMESTAMP) AS effective_date,

        CAST(po.create_date AS TIMESTAMP) AS created_at,
        CAST(po.write_date AS TIMESTAMP) AS updated_at, 

        current_timestamp() AS dwh_loaded_at,
        po.dwh_source_db

    FROM bronze.purchase_order po

    WHERE CAST(po.write_date AS TIMESTAMP) > COALESCE 
    (
        (
            SELECT last_write_date
            FROM silver.load_tracking
            WHERE table_name = 'purchase_order'
        ),
        CAST('1900-01-01' AS TIMESTAMP)
    )

) AS source

ON target.purchase_order_id = source.purchase_order_id

WHEN MATCHED THEN
UPDATE SET
    target.partner_id = source.partner_id,
    target.partner_name = source.partner_name,

    target.company_id = source.company_id,
    target.company_name = source.company_name,

    target.currency_id = source.currency_id,
    target.currency_name = source.currency_name,

    target.user_id = source.user_id,
    target.user_name = source.user_name,

    target.payment_term_id = source.payment_term_id,
    target.payment_term_name = source.payment_term_name,

    target.incoterm_id = source.incoterm_id,
    target.incoterm_name = source.incoterm_name,

    target.fiscal_position_id = source.fiscal_position_id,
    target.fiscal_position_name = source.fiscal_position_name,

    target.picking_type_id = source.picking_type_id,
    target.picking_type_name = source.picking_type_name,

    target.invoice_count = source.invoice_count,

    target.order_name = source.order_name,
    target.priority = source.priority,
    target.origin = source.origin,
    target.partner_ref = source.partner_ref,

    target.order_state = source.order_state,
    target.invoice_status = source.invoice_status,
    target.receipt_status = source.receipt_status,

    target.note = source.note,

    target.amount_untaxed = source.amount_untaxed,
    target.amount_tax = source.amount_tax,
    target.amount_total = source.amount_total,
    target.amount_total_cc = source.amount_total_cc,

    target.currency_rate = source.currency_rate,

    target.locked = source.locked,
    target.acknowledged = source.acknowledged,
    target.receipt_reminder_email = source.receipt_reminder_email,

    target.date_order = source.date_order,
    target.date_approve = source.date_approve,
    target.date_planned = source.date_planned,
    target.effective_date = source.effective_date,

    target.created_at = source.created_at,
    target.updated_at = source.updated_at,

    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;


In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS silver.purchase_order_line
(
    purchase_order_line_id INT,

    order_id INT,
    purchase_order_name STRING, 

    product_id INT,
    product_name STRING,

    product_uom_id INT,
    product_uom_name STRING,

    partner_id INT,
    partner_name STRING,

    company_id INT,
    company_name STRING,

    display_type STRING,
    line_description STRING,
    product_description_variants STRING,
    qty_received_method STRING,
    analytic_distribution STRING,

    product_qty DOUBLE,
    product_uom_qty DOUBLE,
    discount DOUBLE,

    price_unit DOUBLE,
    price_subtotal DOUBLE,
    price_total DOUBLE,
    price_tax DOUBLE,
    technical_price_unit DOUBLE,

    qty_invoiced DOUBLE,
    qty_received DOUBLE,
    qty_received_manual DOUBLE,
    qty_to_invoice DOUBLE,

    is_downpayment BOOLEAN,
    propagate_cancel BOOLEAN,

    date_planned TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
)
USING DELTA;

-- =====================================================
-- MERGE
-- =====================================================

MERGE INTO silver.purchase_order_line AS target

USING
(
    SELECT
        cast( pol.id As int) AS purchase_order_line_id,

        cast( (pol.order_id, '$[0]') As int) AS order_id,
        GET_JSON_OBJECT(pol.order_id, '$[1]') AS purchase_order_name,

        cast( GET_JSON_OBJECT(pol.product_id, '$[0]') As int) AS product_id,
        GET_JSON_OBJECT(pol.product_id, '$[1]') AS product_name,

        cast( GET_JSON_OBJECT(pol.product_uom_id, '$[0]') As int) AS product_uom_id,
        GET_JSON_OBJECT(pol.product_uom_id, '$[1]') AS product_uom_name,

        cast( GET_JSON_OBJECT(pol.partner_id, '$[0]') As int) AS partner_id,
        GET_JSON_OBJECT(pol.partner_id, '$[1]') AS partner_name,

        cast( GET_JSON_OBJECT(pol.company_id, '$[0]') As int) AS company_id,
        GET_JSON_OBJECT(pol.company_id, '$[1]') AS company_name,

        pol.display_type,
        pol.line_description,
        pol.product_description_variants,
        pol.qty_received_method,
        pol.analytic_distribution,

        CAST(pol.product_qty AS DOUBLE) AS product_qty,
        CAST(pol.product_uom_qty AS DOUBLE) AS product_uom_qty,
        CAST(pol.discount AS DOUBLE) AS discount,

        CAST(pol.price_unit AS DOUBLE) AS price_unit,
        CAST(pol.price_subtotal AS DOUBLE) AS price_subtotal,
        CAST(pol.price_total AS DOUBLE) AS price_total,
        CAST(pol.price_tax AS DOUBLE) AS price_tax,
        CAST(pol.technical_price_unit AS DOUBLE) AS technical_price_unit,

        CAST(pol.qty_invoiced AS DOUBLE) AS qty_invoiced,
        CAST(pol.qty_received AS DOUBLE) AS qty_received,
        CAST(pol.qty_received_manual AS DOUBLE) AS qty_received_manual,
        CAST(pol.qty_to_invoice AS DOUBLE) AS qty_to_invoice,

        CAST(pol.is_downpayment AS BOOLEAN) AS is_downpayment,
        CAST(pol.propagate_cancel AS BOOLEAN) AS propagate_cancel,

        CAST(pol.date_planned AS TIMESTAMP) AS date_planned,

        CAST(pol.create_date AS TIMESTAMP) AS created_at,
        CAST(pol.write_date AS TIMESTAMP) AS updated_at,

        current_timestamp() AS dwh_loaded_at,
        pol.dwh_source_db

    FROM bronze.purchase_order_line pol

    WHERE CAST(pol.write_date AS TIMESTAMP) > COALESCE
    (
        (
            SELECT last_write_date
            FROM silver.load_tracking
            WHERE table_name = 'purchase_order_line'
        ),
        CAST('1900-01-01' AS TIMESTAMP)
    )

) AS source

ON target.purchase_order_line_id = source.purchase_order_line_id

WHEN MATCHED THEN
UPDATE SET
    target.order_id = source.order_id,

    target.product_id = source.product_id,
    target.product_name = source.product_name,

    target.product_uom_id = source.product_uom_id,
    target.product_uom_name = source.product_uom_name,

    target.partner_id = source.partner_id,
    target.partner_name = source.partner_name,

    target.company_id = source.company_id,
    target.company_name = source.company_name,

    target.display_type = source.display_type,
    target.line_description = source.line_description,
    target.product_description_variants = source.product_description_variants,
    target.qty_received_method = source.qty_received_method,
    target.analytic_distribution = source.analytic_distribution,

    target.product_qty = source.product_qty,
    target.product_uom_qty = source.product_uom_qty,
    target.discount = source.discount,

    target.price_unit = source.price_unit,
    target.price_subtotal = source.price_subtotal,
    target.price_total = source.price_total,
    target.price_tax = source.price_tax,
    target.technical_price_unit = source.technical_price_unit,

    target.qty_invoiced = source.qty_invoiced,
    target.qty_received = source.qty_received,
    target.qty_received_manual = source.qty_received_manual,
    target.qty_to_invoice = source.qty_to_invoice,

    target.is_downpayment = source.is_downpayment,
    target.propagate_cancel = source.propagate_cancel,

    target.date_planned = source.date_planned,

    target.created_at = source.created_at,
    target.updated_at = source.updated_at,

    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;